# Final Real-Data Benchmark: Standard nanoTabPFN vs NICL

This notebook runs the real forecasting benchmark on the same datasets and protocol used by the repository runbooks, but only executes the two requested models:

- `nanotabpfn_standard`
- `nicl_regression` (NeuralAI / NICL)

It keeps the official final benchmark contract from `configs/forecast_bench_final_3model.json` for datasets, horizons, context/test sizes, and statistics.

In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from tfmplayground.benchmarks.forecasting.adapters import (
    NanoTabPFNForecastAdapter,
    NICLRegressionAdapter,
)
from tfmplayground.benchmarks.forecasting.config import ForecastBenchmarkConfig
from tfmplayground.benchmarks.forecasting.datasets import (
    DEFAULT_DATASET_SPECS,
    load_suite,
)
from tfmplayground.benchmarks.forecasting.metrics import (
    bootstrap_ci,
    relative_improvement,
)
from tfmplayground.benchmarks.forecasting.runner import evaluate_regression
from tfmplayground.interface import NanoTabPFNRegressor

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

/home/mouad/Desktop/dev_projects/tfm_forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
REPO_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
FINAL_CONFIG_PATH = REPO_ROOT / "configs" / "forecast_bench_final_3model.json"
OUTPUT_DIR = REPO_ROOT / "workdir" / "forecast_results_standard_vs_nicl"
PAIRWISE_BASELINE = "nanotabpfn_standard"
PAIRWISE_CANDIDATE = "nicl_regression"


def _load_dotenv_values(path: Path) -> dict[str, str]:
    if not path.exists():
        return {}
    values: dict[str, str] = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def load_eval_cfg(output_dir: Path = OUTPUT_DIR) -> ForecastBenchmarkConfig:
    cfg = ForecastBenchmarkConfig.from_json(FINAL_CONFIG_PATH)
    payload = cfg.to_dict()
    payload["mode"] = "regression"
    payload["output_dir"] = output_dir
    return ForecastBenchmarkConfig.from_dict(payload)


def required_cache_paths(cfg: ForecastBenchmarkConfig) -> dict[str, Path]:
    cache_dir = Path(cfg.datasets.cache_dir)
    return {
        name: cache_dir / DEFAULT_DATASET_SPECS[name].cache_file
        for name in cfg.datasets.dataset_names
    }


def missing_cached_datasets(cfg: ForecastBenchmarkConfig) -> dict[str, Path]:
    cache_map = required_cache_paths(cfg)
    return {name: path for name, path in cache_map.items() if not path.exists()}


def prepare_missing_real_datasets(
    cfg: ForecastBenchmarkConfig, *, force: bool = False
) -> None:
    missing = missing_cached_datasets(cfg)
    if not missing and not force:
        print("All final benchmark datasets are already cached.")
        return

    cmd = [
        sys.executable,
        "scripts/prepare_forecast_datasets.py",
        "--dataset",
        "all",
        "--cache_dir",
        str(cfg.datasets.cache_dir),
    ]
    if force:
        cmd.append("--force")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def check_standard_model_ready() -> str:
    model = NanoTabPFNRegressor(model=None, dist=None, device="cpu")
    return f"loaded {type(model).__name__} via default pretrained checkpoint"


def check_nicl_ready(cfg: ForecastBenchmarkConfig) -> str:
    endpoint = cfg.models.nicl_regression_endpoint
    if not endpoint:
        raise ValueError("NICL regression endpoint is not configured.")

    env_name = cfg.models.nicl_api_key_env
    dotenv_values = _load_dotenv_values(REPO_ROOT / ".env")
    token = os.getenv(env_name)
    source = env_name
    if not token:
        token = os.getenv("NICL_API_TOKEN")
        source = "NICL_API_TOKEN"
    if not token:
        token = dotenv_values.get(env_name) or dotenv_values.get("NICL_API_TOKEN")
        source = ".env"
    if not token:
        raise ValueError(
            f"Missing NICL token in {env_name}, NICL_API_TOKEN, or local .env."
        )
    return f"endpoint configured, token found via {source}"


def build_two_model_adapters(
    cfg: ForecastBenchmarkConfig, *, device: str = "cpu"
) -> dict[str, Any]:
    return {
        "nanotabpfn_standard": NanoTabPFNForecastAdapter(
            name="nanotabpfn_standard",
            model_path=cfg.models.model_standard_ckpt,
            dist_path=cfg.models.model_standard_dist,
            device=device,
        ),
        "nicl_regression": NICLRegressionAdapter(
            api_url=cfg.models.nicl_regression_endpoint,
            timeout_seconds=cfg.models.nicl_timeout_seconds,
            max_retries=cfg.models.nicl_max_retries,
            mode=cfg.models.nicl_regression_mode,
            token_env=cfg.models.nicl_api_key_env,
            model_name=cfg.models.nicl_model,
            proxy_num_classes=cfg.proxy.num_classes,
            min_samples_per_class=cfg.proxy.min_samples_per_class,
        ),
    }


def pairwise_summary(
    rows: pd.DataFrame,
    *,
    cfg: ForecastBenchmarkConfig,
    baseline: str = PAIRWISE_BASELINE,
    candidate: str = PAIRWISE_CANDIDATE,
) -> pd.DataFrame:
    ok_rows = rows.loc[rows["status"] == "ok"].copy()
    key_cols = ["dataset", "series_id", "horizon"]
    base_rows = ok_rows.loc[ok_rows["model"] == baseline]
    cand_rows = ok_rows.loc[ok_rows["model"] == candidate]
    merged = base_rows.merge(cand_rows, on=key_cols, suffixes=("_base", "_cand"))
    if merged.empty:
        return pd.DataFrame(
            columns=[
                "baseline",
                "candidate",
                "metric",
                "mean_improvement",
                "ci_low",
                "ci_high",
                "win_rate",
                "n_pairs",
                "pass",
            ]
        )

    comparisons: list[dict[str, Any]] = []
    for metric_name in cfg.stats.claim_metrics:
        base_metric = merged[f"{metric_name}_base"].to_numpy(dtype=np.float64)
        cand_metric = merged[f"{metric_name}_cand"].to_numpy(dtype=np.float64)
        improvement = relative_improvement(base_metric, cand_metric)
        mean, ci_low, ci_high = bootstrap_ci(
            improvement,
            n_bootstrap=cfg.stats.bootstrap_samples,
            confidence_level=cfg.stats.confidence_level,
            seed=cfg.seed,
        )
        comparisons.append(
            {
                "baseline": baseline,
                "candidate": candidate,
                "metric": metric_name,
                "mean_improvement": float(mean),
                "ci_low": float(ci_low),
                "ci_high": float(ci_high),
                "win_rate": float(np.mean(improvement > 0.0)),
                "n_pairs": int(improvement.size),
                "pass": bool(np.isfinite(ci_low) and ci_low > 0.0),
            }
        )

    summary = pd.DataFrame(comparisons)
    return summary.sort_values(["metric"]).reset_index(drop=True)


def mean_metrics(rows: pd.DataFrame) -> pd.DataFrame:
    ok_rows = rows.loc[rows["status"] == "ok"].copy()
    if ok_rows.empty:
        return pd.DataFrame()
    return (
        ok_rows.groupby("model", as_index=False)[["rmse", "smape", "mase"]]
        .mean()
        .sort_values("model")
        .reset_index(drop=True)
    )


def status_counts(rows: pd.DataFrame) -> pd.DataFrame:
    return (
        rows.groupby(["model", "status"], as_index=False)
        .size()
        .sort_values(["model", "status"])
        .reset_index(drop=True)
    )


def write_outputs(
    *,
    cfg: ForecastBenchmarkConfig,
    rows: pd.DataFrame,
    pairwise: pd.DataFrame,
) -> None:
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    rows_path = output_dir / "regression_rows_standard_vs_nicl.csv"
    pairwise_path = output_dir / "pairwise_summary_standard_vs_nicl.csv"
    meta_path = output_dir / "run_metadata_standard_vs_nicl.json"
    rows.to_csv(rows_path, index=False)
    pairwise.to_csv(pairwise_path, index=False)
    metadata = {
        "config_path": str(FINAL_CONFIG_PATH),
        "datasets": list(cfg.datasets.dataset_names),
        "models_ran": ["nanotabpfn_standard", "nicl_regression"],
        "protocol": {
            "horizons": list(cfg.protocol.horizons),
            "context_rows": cfg.protocol.context_rows,
            "test_rows": cfg.protocol.test_rows,
            "max_feature_lag": cfg.protocol.max_feature_lag,
            "explicit_lags": list(cfg.protocol.explicit_lags),
            "num_kernels": cfg.protocol.num_kernels,
            "add_mask_channels": cfg.protocol.add_mask_channels,
        },
        "nicl": {
            "mode": cfg.models.nicl_regression_mode,
            "endpoint": cfg.models.nicl_regression_endpoint,
            "model": cfg.models.nicl_model,
            "token_env": cfg.models.nicl_api_key_env,
        },
        "stats": {
            "bootstrap_samples": cfg.stats.bootstrap_samples,
            "confidence_level": cfg.stats.confidence_level,
            "claim_metrics": list(cfg.stats.claim_metrics),
        },
    }
    meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(rows_path)
    print(pairwise_path)
    print(meta_path)

In [3]:
cfg = load_eval_cfg()

print("Repo root:", REPO_ROOT)
print("Final config:", FINAL_CONFIG_PATH)
print("Output dir:", cfg.output_dir)
print("Datasets:", list(cfg.datasets.dataset_names))
print("Protocol horizons:", list(cfg.protocol.horizons))
print("NICL mode:", cfg.models.nicl_regression_mode)
print("NICL endpoint:", cfg.models.nicl_regression_endpoint)

missing = missing_cached_datasets(cfg)
print("Missing cached datasets:")
for name, path in missing.items():
    print(f"  - {name}: {path}")
if not missing:
    print("  none")

print("Standard readiness:", check_standard_model_ready())
print("NICL readiness:", check_nicl_ready(cfg))

Repo root: /home/mouad/Desktop/dev_projects/tfm_forecasting
Final config: /home/mouad/Desktop/dev_projects/tfm_forecasting/configs/forecast_bench_final_3model.json
Output dir: /home/mouad/Desktop/dev_projects/tfm_forecasting/workdir/forecast_results_standard_vs_nicl
Datasets: ['exchange_rate', 'ettm1', 'm4_weekly', 'tourism_monthly']
Protocol horizons: [1, 3, 6, 12]
NICL mode: quantized_proxy
NICL endpoint: https://prediction.neuralk-ai.com/predict
Missing cached datasets:
  - ettm1: workdir/forecast_data/ettm1.npz
  - m4_weekly: workdir/forecast_data/m4_weekly.npz
  - tourism_monthly: workdir/forecast_data/tourism_monthly.npz
No cached model found, downloading model checkpoint.
No cached bucket edges found, downloading bucket edges.
Standard readiness: loaded NanoTabPFNRegressor via default pretrained checkpoint
NICL readiness: endpoint configured, token found via NEURALK_API_KEY


In [4]:
# Run this cell once if m4_weekly or tourism_monthly are missing.
# It reuses existing cached files unless force=True.

prepare_missing_real_datasets(cfg, force=False)
missing_cached_datasets(cfg)

Running: /home/mouad/Desktop/dev_projects/tfm_forecasting/.venv/bin/python scripts/prepare_forecast_datasets.py --dataset all --cache_dir workdir/forecast_data
m4_weekly: written -> workdir/forecast_data/m4_weekly.npz
tourism_monthly: written -> workdir/forecast_data/tourism_monthly.npz


{'ettm1': PosixPath('workdir/forecast_data/ettm1.npz'),
 'm4_weekly': PosixPath('workdir/forecast_data/m4_weekly.npz'),
 'tourism_monthly': PosixPath('workdir/forecast_data/tourism_monthly.npz')}

In [5]:
suite = load_suite(cfg)
for name, bundle in suite.items():
    print(
        name,
        "skipped=",
        bundle.skipped,
        "shape=",
        bundle.series.shape,
        "reason=",
        bundle.skip_reason,
    )

adapters = build_two_model_adapters(cfg, device="cpu")
rows = evaluate_regression(cfg, suite=suite, adapters=adapters, device="cpu")

pairwise = pairwise_summary(rows, cfg=cfg)
means = mean_metrics(rows)
statuses = status_counts(rows)

write_outputs(cfg=cfg, rows=rows, pairwise=pairwise)

exchange_rate skipped= False shape= (8, 7588) reason= None
ettm1 skipped= False shape= (7, 69680) reason= None
m4_weekly skipped= True shape= (0, 0) reason= Prepare cached npz with scripts/prepare_forecast_datasets.py (key='series').
tourism_monthly skipped= True shape= (0, 0) reason= Prepare cached npz with scripts/prepare_forecast_datasets.py (key='series').
/home/mouad/Desktop/dev_projects/tfm_forecasting/workdir/forecast_results_standard_vs_nicl/regression_rows_standard_vs_nicl.csv
/home/mouad/Desktop/dev_projects/tfm_forecasting/workdir/forecast_results_standard_vs_nicl/pairwise_summary_standard_vs_nicl.csv
/home/mouad/Desktop/dev_projects/tfm_forecasting/workdir/forecast_results_standard_vs_nicl/run_metadata_standard_vs_nicl.json


In [6]:
print("Status counts")
display(statuses)

print("Mean regression metrics")
display(means)

print("Pairwise improvement of NICL vs standard nanoTabPFN")
display(pairwise)

print("Top skip reasons, if any")
display(
    rows.loc[
        rows["status"] == "skipped", ["model", "dataset", "series_id", "skip_reason"]
    ]
    .sort_values(["model", "dataset", "series_id"])
    .head(50)
)

Status counts


,model,status,size
0,nanotabpfn_standard,ok,60
1,nanotabpfn_standard,skipped,2
2,nicl_regression,skipped,17


Mean regression metrics


,model,rmse,smape,mase
0,nanotabpfn_standard,1.6982,0.38063,14.450896


Pairwise improvement of NICL vs standard nanoTabPFN


,baseline,candidate,metric,mean_improvement,ci_low,ci_high,win_rate,n_pairs,pass


Top skip reasons, if any


,model,dataset,series_id,skip_reason
75,nanotabpfn_standard,m4_weekly,-1,Prepare cached npz with scripts/prepare_foreca...
77,nanotabpfn_standard,tourism_monthly,-1,Prepare cached npz with scripts/prepare_foreca...
44,nicl_regression,ettm1,0,NICL request failed after retries: 405 Client ...
49,nicl_regression,ettm1,1,NICL request failed after retries: 405 Client ...
54,nicl_regression,ettm1,2,NICL request failed after retries: 405 Client ...
59,nicl_regression,ettm1,3,NICL request failed after retries: 405 Client ...
64,nicl_regression,ettm1,4,NICL request failed after retries: 405 Client ...
69,nicl_regression,ettm1,5,NICL request failed after retries: 405 Client ...
74,nicl_regression,ettm1,6,NICL request failed after retries: 405 Client ...
4,nicl_regression,exchange_rate,0,NICL request failed after retries: 405 Client ...


In [9]:
rows["skip_reason"].iloc[4]

'NICL request failed after retries: 405 Client Error: Not Allowed for url: https://prediction.neuralk-ai.com/predict'